# 03 — Proditec: surrogate benchmarking and training

Benchmarks 12 scikit-learn/XGBoost regressors plus CatBoost on the surrogate-ready dataset
from notebook 02 using an 80/20 stratified split, then fixes CatBoost as the operational
surrogate (Section 3.4) with the paper's exact hyperparameters (5000 iterations, learning rate
0.05, depth 8, `l2_leaf_reg=3.0`, early stopping after 200 rounds). Produces the
development-split Proditec row of **Table 2** (Section 4.3).

The *primary* generalisation estimate reported in the paper (5-fold GroupKFold cross-validation
grouped by exact configuration signature, R2 = 0.934 +/- 0.008) is **not** computed here — it
lives in `shared/07_cross_dataset_revision_analysis.ipynb` (section S10).

**Public-release fix:** the final cell now saves the model under the dataset-prefixed name
`proditec_surrogate_catboost.cbm` and additionally exports `proditec_surrogate_metadata.json`
(features/categorical/numeric/target/RMSE/MAE/R2/seed), mirroring the MVTec twin notebook. The
original cell saved only an unprefixed `surrogate_catboost.cbm` with no metadata file, which
meant a from-scratch re-run (with no manually-copied artifacts) would fail the model-file
assertion in notebook 04 and silently fall back to a hardcoded feature schema in the
cross-dataset revision notebook.


# Benchmarking Surrogate Models

In [ ]:
# ============================================================
# Imports & Global Setup
# ============================================================
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import dice_ml
import random
import json
import datetime

from pathlib import Path
from dice_ml import Dice
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.base import clone

from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

from sklearn.ensemble import (
    RandomForestRegressor,
    ExtraTreesRegressor,
    GradientBoostingRegressor,
    HistGradientBoostingRegressor,
    AdaBoostRegressor,
)
from sklearn.tree import DecisionTreeRegressor
from sklearn.linear_model import BayesianRidge, ElasticNet, HuberRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR

from xgboost import XGBRegressor
from catboost import CatBoostRegressor

SEED = 42

## 1. Load Surrogate Dataset

In [ ]:
# -----------------------------
# 1) Load
# -----------------------------
CSV_PATH = "surrogate_ready_dataset/patchcore_surrogate_dataset_xgb.csv"
df = pd.read_csv(CSV_PATH)

print("Loaded:", CSV_PATH)
print("Shape:", df.shape)
display(df.head())

## 2. Train/Test Split (Stratified for Regression)

In [ ]:
# -----------------------------
# 2) Split 80/20 (stratified for regression)
# -----------------------------
TARGET = "value"
X = df.drop(columns=[TARGET])
y = df[TARGET]

y_series = y.reset_index(drop=True)

is_zero = (y_series == 0.0)
strat = pd.Series(index=y_series.index, dtype="object")
strat[is_zero] = "zero"

n_bins = 10
nonzero = y_series[~is_zero]
qbins = pd.qcut(nonzero, q=n_bins, duplicates="drop")
strat.loc[nonzero.index] = qbins.astype(str)

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=SEED,
    shuffle=True,
    stratify=strat
)

print("Train:", X_train.shape, y_train.shape)
print("Test: ", X_test.shape,  y_test.shape)
print("Zero rate train:", float((y_train == 0.0).mean()))
print("Zero rate test: ", float((y_test == 0.0).mean()))


## 3. Feature Schema & Preprocessing

In [ ]:
# -----------------------------
# 3) Feature schema
# -----------------------------
categorical_features = [
    "params_backbone",
    "params_batch_size",
    "params_center_crop_key",
    "params_image_size_key",
    "params_layers_key",
    "params_max_patches_per_image",
    "params_pre_trained",
    "params_reduction",
    "params_soft_corruption_level",
]

numeric_features = [
    "params_coreset_sampling_ratio",
    "params_num_neighbors",
    "params_soft_review_budget",
    "params_soft_train_fraction",
]

assert set(categorical_features + numeric_features) == set(X_train.columns), "Feature lists mismatch"

print("Categorical:", len(categorical_features), categorical_features)
print("Numeric:", len(numeric_features), numeric_features)


In [ ]:
# -----------------------------
# 4) Preprocessing (sklearn models)
# -----------------------------
preprocess = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
        ("num", "passthrough", numeric_features),
    ],
    remainder="drop",
)


## 4. Baseline Benchmark (sklearn models)

In [ ]:
# -----------------------------
# 5) Benchmark utilities
# -----------------------------
def eval_regression(y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)
    rmse = float(np.sqrt(mse))
    mae = float(mean_absolute_error(y_true, y_pred))
    r2 = float(r2_score(y_true, y_pred))
    return rmse, mae, r2

def benchmark(models, X_train, y_train, X_test, y_test, preprocess):
    rows = []
    fitted = {}

    for name, est in models.items():
        pipe = Pipeline([("prep", preprocess), ("model", clone(est))])

        try:
            t0 = time.time()
            pipe.fit(X_train, y_train)
            fit_sec = time.time() - t0

            y_pred = pipe.predict(X_test)
            rmse, mae, r2 = eval_regression(y_test, y_pred)

            rows.append({
                "model": name,
                "rmse": rmse,
                "mae": mae,
                "r2": r2,
                "fit_sec": fit_sec,
                "status": "ok",
            })
            fitted[name] = pipe

        except Exception as e:
            rows.append({
                "model": name,
                "rmse": np.nan,
                "mae": np.nan,
                "r2": np.nan,
                "fit_sec": np.nan,
                "status": f"fail: {type(e).__name__}",
            })
            print(f"[WARN] {name} failed -> {type(e).__name__}: {e}")

    res = pd.DataFrame(rows).sort_values(["rmse", "mae"], ascending=[True, True]).reset_index(drop=True)
    return res, fitted


In [ ]:
# -----------------------------
# 6) Model zoo (sklearn benchmark)
# -----------------------------
models = {
    "HistGBDT": HistGradientBoostingRegressor(random_state=SEED),
    "RandomForest": RandomForestRegressor(n_estimators=400, random_state=SEED, n_jobs=-1),
    "ExtraTrees": ExtraTreesRegressor(n_estimators=800, random_state=SEED, n_jobs=-1),
    "GBDT": GradientBoostingRegressor(random_state=SEED),

    "XGBoost": XGBRegressor(
        n_estimators=800,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.9,
        colsample_bytree=0.9,
        objective="reg:squarederror",
        random_state=SEED,
        n_jobs=-1,
    ),

    "AdaBoost": AdaBoostRegressor(
        estimator=DecisionTreeRegressor(max_depth=4, random_state=SEED),
        n_estimators=400,
        learning_rate=0.05,
        random_state=SEED,
    ),
    "DecisionTree": DecisionTreeRegressor(random_state=SEED),
    "KNN": KNeighborsRegressor(n_neighbors=25, weights="distance"),
    "SVR(RBF)": SVR(C=10.0, epsilon=0.02, gamma="scale"),

    "BayesianRidge": BayesianRidge(),
    "ElasticNet": ElasticNet(alpha=1e-3, l1_ratio=0.2, random_state=SEED),
    "Huber": HuberRegressor(),
}

print("Models:", list(models.keys()))


In [ ]:
# -----------------------------
# 7) Run sklearn benchmark
# -----------------------------
results, fitted_pipes = benchmark(models, X_train, y_train, X_test, y_test, preprocess)

display(results)
best_sklearn = results.loc[results["status"] == "ok"].iloc[0]["model"]
print("Best (sklearn benchmark):", best_sklearn)


In [ ]:
# -----------------------------
# 8) CatBoost setup (native categorical handling)
# -----------------------------

# Copy to avoid mutating X_train / X_test used by sklearn pipelines
X_train_cb = X_train.copy()
X_test_cb = X_test.copy()

# Ensure categoricals are strings (CatBoost categorical processing expects this)
for c in categorical_features:
    X_train_cb[c] = X_train_cb[c].astype(str)
    X_test_cb[c] = X_test_cb[c].astype(str)

# CatBoost needs categorical feature indices
cat_features_idx = [X_train_cb.columns.get_loc(c) for c in categorical_features]

print("CatBoost categorical feature indices:", cat_features_idx)
print("CatBoost categorical features:", categorical_features)


In [ ]:
# -----------------------------
# 9) Train CatBoost (early stopping)
# -----------------------------
cb = CatBoostRegressor(
    loss_function="RMSE",
    eval_metric="RMSE",
    iterations=5000,          # early stopping will select the best iteration
    learning_rate=0.05,
    depth=8,
    l2_leaf_reg=3.0,
    random_seed=SEED,
    verbose=200,
)

t0 = time.time()
cb.fit(
    X_train_cb, y_train,
    cat_features=cat_features_idx,
    eval_set=(X_test_cb, y_test),
    use_best_model=True,
    early_stopping_rounds=200,
)
cb_fit_sec = time.time() - t0

print("CatBoost fit_sec:", cb_fit_sec)


In [ ]:
# -----------------------------
# 10) Evaluate CatBoost + merge into benchmark table
# -----------------------------
y_pred_cb = cb.predict(X_test_cb)
rmse_cb, mae_cb, r2_cb = eval_regression(y_test, y_pred_cb)

print(f"CatBoost(native) -> RMSE: {rmse_cb:.6f} | MAE: {mae_cb:.6f} | R²: {r2_cb:.6f}")

cb_row = pd.DataFrame([{
    "model": "CatBoost(native)",
    "rmse": rmse_cb,
    "mae": mae_cb,
    "r2": r2_cb,
    "fit_sec": cb_fit_sec,
    "status": "ok",
}])

results_all = (
    pd.concat([results, cb_row], ignore_index=True)
      .sort_values(["rmse", "mae"], ascending=[True, True])
      .reset_index(drop=True)
)

display(results_all)

best_overall = results_all.loc[results_all["status"] == "ok"].iloc[0]["model"]
print("Best (overall incl. CatBoost):", best_overall)


## 5. Final Surrogate: CatBoost (Native Categoricals)

In [ ]:
# -----------------------------
# 11) Select final surrogate (fixed choice)
# -----------------------------
FINAL_MODEL = "CatBoost(native)"
print("Selected surrogate:", FINAL_MODEL)


In [ ]:
# -----------------------------
# 12) CatBoost predictions (test set)
# -----------------------------
y_true = y_test.values if hasattr(y_test, "values") else y_test
y_pred = np.asarray(cb.predict(X_test_cb))

residuals = y_true - y_pred
abs_err = np.abs(residuals)

print("CatBoost(native) evaluation on test")
print("y_true: min/max =", float(np.min(y_true)), float(np.max(y_true)))
print("y_pred: min/max =", float(np.min(y_pred)), float(np.max(y_pred)))


## 6. Surrogate Diagnostics (CatBoost)

In [ ]:
# -----------------------------
# 13) Residuals vs Predicted
# -----------------------------
plt.figure(figsize=(6.5, 4.5))
plt.scatter(y_pred, residuals, alpha=0.35)
plt.axhline(0, linestyle="--")
plt.xlabel("Predicted value")
plt.ylabel("Residual (true - pred)")
plt.title(f"Residuals vs Predicted — {best_overall}")
plt.tight_layout()
plt.show()


In [ ]:
# -----------------------------
# 14) Predicted vs True (CatBoost)
# -----------------------------
plt.figure(figsize=(5.8, 5.8))
plt.scatter(y_true, y_pred, alpha=0.35)
plt.plot([0, 1], [0, 1], "--")
plt.xlabel("True value")
plt.ylabel("Predicted value")
plt.title("CatBoost(native): Predicted vs True")
plt.tight_layout()
plt.show()


In [ ]:
# -----------------------------
# 16) Residual histogram (CatBoost)
# -----------------------------
plt.figure(figsize=(7.0, 4.8))
plt.hist(residuals, bins=40)
plt.axvline(0, linestyle="--")
plt.xlabel("Residual (true - pred)")
plt.ylabel("Count")
plt.title("CatBoost(native): Residual distribution")
plt.tight_layout()
plt.show()


In [ ]:
# -----------------------------
# 17) Error by target region (CatBoost)
# -----------------------------
bins = pd.qcut(pd.Series(y_true), q=10, duplicates="drop")
err_by_bin = (
    pd.DataFrame({"bin": bins.astype(str), "abs_err": abs_err})
      .groupby("bin")["abs_err"]
      .agg(mean="mean", median="median", count="count")
      .reset_index()
)

display(err_by_bin)

plt.figure(figsize=(9.5, 4.2))
plt.plot(range(len(err_by_bin)), err_by_bin["mean"], marker="o")
plt.xticks(range(len(err_by_bin)), err_by_bin["bin"], rotation=45, ha="right")
plt.ylabel("Mean absolute error")
plt.title("CatBoost(native): Binned MAE across target distribution")
plt.tight_layout()
plt.show()


In [ ]:
# -----------------------------
# 18) Calibration (bin means: true vs predicted)
# -----------------------------
cal_bins = pd.qcut(pd.Series(y_pred), q=10, duplicates="drop")
cal = (
    pd.DataFrame({"bin": cal_bins.astype(str), "y_true": y_true, "y_pred": y_pred})
      .groupby("bin")
      .agg(y_true_mean=("y_true", "mean"), y_pred_mean=("y_pred", "mean"), n=("y_true", "size"))
      .reset_index()
)

display(cal)

plt.figure(figsize=(6.2, 6.2))
plt.scatter(cal["y_pred_mean"], cal["y_true_mean"])
plt.plot([0, 1], [0, 1], "--")
plt.xlabel("Mean predicted (bin)")
plt.ylabel("Mean true (bin)")
plt.title("CatBoost(native): Calibration by predicted quantiles")
plt.tight_layout()
plt.show()


## Export CFs + metadata

In [ ]:
# 19) Save surrogate model (CatBoost) + metadata
#
# Public-release fix: the original cell only called cb.save_model(), with no
# metadata JSON export and an unprefixed filename ("surrogate_catboost.cbm").
# The candidate-generation notebook (04) and the cross-dataset revision
# notebook both expect a dataset-prefixed model AND a metadata JSON (mirroring
# the MVTec twin's mvtec_surrogate_catboost.cbm / mvtec_surrogate_metadata.json),
# so this cell now writes both under the names the rest of the pipeline reads.
OUT_DIR = Path("surrogate_pkl_cfs_metadata")
OUT_DIR.mkdir(parents=True, exist_ok=True)

model_path = OUT_DIR / "proditec_surrogate_catboost.cbm"
cb.save_model(str(model_path))

try:
    best_iteration = int(cb.get_best_iteration())
except Exception:
    best_iteration = None

meta_path = OUT_DIR / "proditec_surrogate_metadata.json"
metadata = {
    "dataset": str(CSV_PATH),
    "n_rows": int(len(df)),
    "n_train": int(len(X_train)),
    "n_test": int(len(X_test)),
    "target": TARGET,
    "features": categorical_features + numeric_features,
    "categorical_features": categorical_features,
    "numeric_features": numeric_features,
    "final_model": "CatBoost(native)",
    "rmse": float(rmse_cb),
    "mae": float(mae_cb),
    "r2": float(r2_cb),
    "fit_sec": float(cb_fit_sec),
    "best_iteration": best_iteration,
    "seed": SEED,
}
with open(meta_path, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2)

print(" - model:   ", model_path)
print(" - metadata:", meta_path)
